funções de apoio, apenas em caso de necessidade e d caracter genérico, exeplo seria elementos auxiliares como funções de selecionar com interface uma pasta, ou de resolver informações de caracter auxiliar que n seja considerados de elementos necessários para a função funcionar, são apenas comuns entre as funções e q podem dar jeito. 

# Bibliotecas utilizadas

> Funções genéricas de apoio — importações necessárias para os restantes notebooks da biblioteca comum.

In [ ]:
# Permite visualizar figuras diretamente no notebook
%matplotlib inline

# ==========================================================
# BIBLIOTECAS UTILIZADAS
# ==========================================================

import io
from pathlib import Path

import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

### 1. Função `converter_para_grayscale()`

Converte imagens RGB/RGBA para escala de cinzentos quando necessário.

- preserva as dimensões espaciais (H, W);
- utiliza a fórmula de luminância ponderada ensinada na UC;
- se a imagem já for 2D, devolve-a sem alteração.

In [ ]:
# ==========================================================
# 1. CONVERSÃO PARA ESCALA DE CINZENTOS
# ==========================================================

def converter_para_grayscale(img):
    """
    Converte imagem RGB/RGBA para grayscale preservando dimensões (H, W).
    """

    # Se já for imagem 2D, não é necessário converter
    if img.ndim == 2:
        return img

    # Extrair canais (assumindo ordem RGB)
    canal_vermelho = img[:, :, 0]
    canal_verde = img[:, :, 1]
    canal_azul = img[:, :, 2]

    # Fórmula de luminância (valores usados na UC)
    imagem_cinza = 0.299 * canal_vermelho + 0.587 * canal_verde + 0.114 * canal_azul

    return imagem_cinza

### 2. Função `garantir_uint8_manual()`

Garante que a imagem está no formato `uint8` com intensidades em [0, 255].

- processo explícito: escalar → arredondar → limitar → converter tipo;
- evita saturar valores fora do intervalo válido.

In [ ]:
# ==========================================================
# 2. GARANTIR FORMATO UINT8 (PASSO A PASSO)
# ==========================================================

def garantir_uint8_manual(img):
    """
    Converte a imagem para uint8 no intervalo [0, 255].
    """

    # Passo 1 — se já for uint8, devolver cópia
    if img.dtype == np.uint8:
        return img.copy()

    # Passo 2 — trabalhar em float para evitar perda intermédia
    imagem_float = img.astype(np.float64)

    # Passo 3 — se valores estiverem normalizados entre 0 e 1, escalar
    valor_maximo = imagem_float.max()
    if valor_maximo <= 1.0:
        imagem_escalada = imagem_float * 255.0
    else:
        imagem_escalada = imagem_float.copy()

    # Passo 4 — arredondar para o inteiro mais próximo
    imagem_arredondada = np.round(imagem_escalada)

    # Passo 5 — garantir intervalo [0, 255]
    imagem_limitada = np.clip(imagem_arredondada, 0, 255)

    # Passo 6 — converter para uint8
    imagem_uint8 = imagem_limitada.astype(np.uint8)

    return imagem_uint8

### 3. Função `preparar_imagem_uint8_cinza()`

Combina as etapas de preparação mais frequentes: grayscale + `uint8`.

- função de apoio reutilizada pelos outros notebooks;
- não altera o conteúdo sem necessidade (apenas normaliza representação).

In [ ]:
# ==========================================================
# 3. PREPARAR IMAGEM (GRAYSCALE + UINT8)
# ==========================================================

def preparar_imagem_uint8_cinza(img):
    """
    Prepara imagem para processamento clássico: grayscale e uint8.
    """

    imagem_cinza = converter_para_grayscale(img)
    imagem_uint8 = garantir_uint8_manual(imagem_cinza)

    return imagem_uint8

### 4. Função `carregar_imagem()`

Carrega um ficheiro de imagem do disco e devolve array preparado (`uint8`, grayscale).

- utiliza `matplotlib.image` (como nos worksheets);
- caminho pode ser `str` ou `Path`.

In [ ]:
# ==========================================================
# 4. CARREGAR IMAGEM A PARTIR DE CAMINHO
# ==========================================================

def carregar_imagem(caminho):
    """
    Lê imagem do disco e devolve uint8 em escala de cinzentos.
    """

    caminho_ficheiro = Path(caminho)

    if not caminho_ficheiro.is_file():
        raise FileNotFoundError(f"Ficheiro não encontrado: {caminho_ficheiro}")

    # Leitura com matplotlib (formato do worksheet)
    imagem_lida = mpimg.imread(caminho_ficheiro)

    # Preparar para processamento
    imagem_preparada = preparar_imagem_uint8_cinza(imagem_lida)

    return imagem_preparada

### 5. Função `mostrar_imagens()`

Apresentação lado a lado de até 4 imagens (apoio visual durante o TP).

- colormap `gray` automático para imagens 2D;
- títulos opcionais por imagem.

In [ ]:
# ==========================================================
# 5. APRESENTAÇÃO DE IMAGENS (ATÉ 4)
# ==========================================================

def mostrar_imagens(imagens, titulos=None, mostrar_eixos=False, cmap="gray"):
    """
    Mostra entre 1 e 4 imagens na horizontal.
    """

    if not isinstance(imagens, list):
        raise ValueError("O parâmetro 'imagens' deve ser uma lista.")

    numero_imagens = len(imagens)
    if numero_imagens < 1 or numero_imagens > 4:
        raise ValueError("A função aceita entre 1 e 4 imagens.")

    plt.figure(figsize=(5 * numero_imagens, 5))

    for indice in range(numero_imagens):
        plt.subplot(1, numero_imagens, indice + 1)
        imagem_atual = imagens[indice]

        if imagem_atual.ndim == 2:
            plt.imshow(imagem_atual, cmap=cmap)
        else:
            plt.imshow(imagem_atual)

        if titulos is not None and indice < len(titulos):
            plt.title(titulos[indice])

        if not mostrar_eixos:
            plt.axis("off")

    plt.tight_layout()
    plt.show()

### 6. Função `analisar_metadados()`

Mostra informação auxiliar sobre o array da imagem (dimensão, tipo, min/max).

- útil para validar carregamento antes de aplicar filtros ou transformações.

In [ ]:
# ==========================================================
# 6. METADADOS DA IMAGEM (INFORMAÇÃO AUXILIAR)
# ==========================================================

def analisar_metadados(img, nome="imagem"):
    """
    Imprime dimensões, tipo de dados e intervalo de intensidades.
    """

    print(f"--- Metadados: {nome} ---")
    print(f"Dimensão (shape): {img.shape}")
    print(f"Tipo (dtype): {img.dtype}")

    if img.size == 0:
        print("Array vazio.")
        return

    valor_minimo = img.min()
    valor_maximo = img.max()
    print(f"Intensidade mínima: {valor_minimo}")
    print(f"Intensidade máxima: {valor_maximo}")

### 7. Função `criar_seletor_pasta()`

Interface simples (texto + botão) para indicar uma pasta no Jupyter.

- apoio à exploração de datasets locais;
- devolve o caminho escolhido após confirmação.

In [ ]:
# ==========================================================
# 7. SELECIONAR PASTA (INTERFACE NO NOTEBOOK)
# ==========================================================

def criar_seletor_pasta(descricao="Caminho da pasta"):
    """
    Cria widgets para o utilizador indicar uma pasta.
    Devolve o Path após clicar em Confirmar.
    """

    campo_caminho = widgets.Text(
        description=descricao,
        placeholder=str(Path.cwd()),
        style={"description_width": "initial"},
    )
    botao_confirmar = widgets.Button(description="Confirmar pasta")
    area_resultado = widgets.Output()

    caminho_escolhido = {"valor": None}

    def ao_clicar(_):
        with area_resultado:
            area_resultado.clear_output()
            caminho = Path(campo_caminho.value).expanduser().resolve()
            caminho_escolhido["valor"] = caminho
            print(f"Pasta selecionada: {caminho}")

    botao_confirmar.on_click(ao_clicar)

    display(campo_caminho, botao_confirmar, area_resultado)

    return caminho_escolhido